In [2]:
%load_ext autoreload
%autoreload 2

import os
print(os.getcwd())
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.optim import Adam,AdamW
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import pickle

from train_re import AdaptiveTAPEandDiffusion2, alternate_training_earlyStop, evaluation, adaptive_stage_domain9
from utils import simdatset, reproducibility, calculate_evaluation_metrics

batch_size = 256
reproducibility(2024)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/disk1/user/liaoshuilin/project/35.TAPE_EXO/assay_diffusion


In [13]:
item_values = ["T3C2", "T3C3", "T10C1", "T10C2", "T10C3"]
# item_values = ["T3C2"]

for item in item_values:

    out_pth = "../result/model_variTiss/" + item + "_"
    print(out_pth)
    with open(f'../result/data_variTiss/Stim_data_' + item + '.pkl', 'rb') as file:
        loaded_data = pickle.load(file)
    GTE_x_train = loaded_data['GTE_x_train']
    print(GTE_x_train.shape)
    GTE_x_val = loaded_data['GTE_x_val']
    GTE_x_test = loaded_data['GTE_x_test']
    GTE_y_train = loaded_data['GTE_y_train']
    print(GTE_y_train.shape)
    GTE_y_val = loaded_data['GTE_y_val']
    GTE_y_test = loaded_data['GTE_y_test']
    HPA_x = loaded_data['HPA_x']
    HPA_y = loaded_data['HPA_y']
    print(HPA_x.shape)
    print(HPA_y.shape)

    model = AdaptiveTAPEandDiffusion2(GTE_x_train.shape[1], GTE_y_train.shape[1],2, T=2000).to(device)
    optimizer_main = AdamW([
        {'params': model.encoder.parameters()},
        {'params': model.predictor.parameters()},
        {'params': model.decoder.parameters()}], lr=1e-4)
    optimizer_diffusion =AdamW(model.ref_creator.parameters(), lr=1e-3)
    optimizer_all = AdamW(model.parameters(),1e-4)
    epochs_main = 400 
    epochs_diffusion = 1500
    train_loader = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(simdatset(GTE_x_val, GTE_y_val), batch_size=batch_size, shuffle=False)
    model, main_loss, diffloss = alternate_training_earlyStop(model, train_loader, val_loader, optimizer_main, optimizer_diffusion, 
                                                    epochs_main, epochs_diffusion, device='cpu',
                                                    patience=5, early_stop_start=400, early_stop_interval=50)
    torch.save(model, out_pth + "model_stage1.pth")
    train_loader2 = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=False)
    x_recon_tr, f_tr, z_tr  = evaluation(train_loader2, model, device=device)
    print(f_tr.shape)
    sigmatrix = np.linalg.pinv(f_tr) @ x_recon_tr 
    pd.DataFrame(sigmatrix).to_csv(out_pth + 'sigmatrix.csv', index=True, header=True)

    x_recon_HPA, f_HPA, z_HPA, model2 = adaptive_stage_domain9(x=HPA_x, model_name=out_pth + "model_stage1", mode = 'overall5', steps=10, max_iter=40, device=device, sigmatrix = sigmatrix)
    pd.DataFrame(x_recon_HPA).to_csv(out_pth + 'HPA_DADA_x_recon.csv', index=False, header=False)
    pd.DataFrame(f_HPA).to_csv(out_pth + 'HPA_DADA_y.csv', index=False, header=False)
    rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = HPA_x, out_pth = out_pth + "HPA_DADA_x_")
    print(f"DADA\nRMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")
    rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_HPA, input_X = HPA_y, out_pth = out_pth + "HPA_DADA_y_")
    print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")
    torch.save(model2, out_pth + "model_stage2.pth")

/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_variTiss/T3C2_
(4000, 18757)
(4000, 3)
(50, 18757)
(50, 3)
Epoch [1/400], Main Loss: 0.3361
Epoch [2/400], Main Loss: 0.1038
Epoch [3/400], Main Loss: 0.0516
Epoch [4/400], Main Loss: 0.0395
Epoch [5/400], Main Loss: 0.0332
Epoch [6/400], Main Loss: 0.0299
Epoch [7/400], Main Loss: 0.0278
Epoch [8/400], Main Loss: 0.0252
Epoch [9/400], Main Loss: 0.0244
Epoch [10/400], Main Loss: 0.0226
Epoch [11/400], Main Loss: 0.0203
Epoch [12/400], Main Loss: 0.0204
Epoch [13/400], Main Loss: 0.0199
Epoch [14/400], Main Loss: 0.0184
Epoch [15/400], Main Loss: 0.0176
Epoch [16/400], Main Loss: 0.0176
Epoch [17/400], Main Loss: 0.0169
Epoch [18/400], Main Loss: 0.0159
Epoch [19/400], Main Loss: 0.0151
Epoch [20/400], Main Loss: 0.0151
Epoch [21/400], Main Loss: 0.0151
Epoch [22/400], Main Loss: 0.0152
Epoch [23/400], Main Loss: 0.0138
Epoch [24/400], Main Loss: 0.0142
Epoch [25/400], Main Loss: 0.0135
Epoch [26/400], Main Loss: 0.0134
Epoch [27

In [ ]:
%load_ext autoreload
%autoreload 2
from utils import calculate_evaluation_metrics

item_values = ["T3C1", "T3C2", "T3C3", "T10C1", "T10C2", "T10C3"]
# item_values = ["T3C2"]

for item in item_values:

    out_pth = "/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_variTiss/" + item + "_"
    print(out_pth)
    with open(f'../result/data_variTiss/Stim_data_' + item + '.pkl', 'rb') as file:
        loaded_data = pickle.load(file)
    GTE_x_train = loaded_data['GTE_x_train']
    print(GTE_x_train.shape)
    GTE_x_val = loaded_data['GTE_x_val']
    GTE_x_test = loaded_data['GTE_x_test']
    GTE_y_train = loaded_data['GTE_y_train']
    print(GTE_y_train.shape)
    GTE_y_val = loaded_data['GTE_y_val']
    GTE_y_test = loaded_data['GTE_y_test']
    HPA_x = loaded_data['HPA_x']
    HPA_y = loaded_data['HPA_y']
    print(HPA_x.shape)
    print(HPA_y.shape)

    # pd.DataFrame(GTE_y_test).to_csv(out_pth + 'GTE_real_y.csv', index=False, header=False)
    pd.DataFrame(HPA_y).to_csv(out_pth + 'HPA_real_y.csv', index=False, header=False)

    x_recon_HPA = pd.read_csv(out_pth + 'HPA_DADA_x_recon.csv', header=None)
    f_HPA =  pd.read_csv(out_pth + 'HPA_DADA_y.csv', header=None)

    rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = HPA_x)
    print(f"DADA\nRMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")
    rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = f_HPA, input_X = HPA_y)
    print(f"RMSE of test frac: {average_rmse:.4f}\nPCC of frac: {average_pearson_corr:.4f}\nMAE of frac: {average_mae:.4f}\nCCC of frac: {average_ccc:.4f}")